In [28]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [29]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")


c:\Users\Admin\anaconda3\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Pleas

In [30]:
# loading large datasets using dataset class
from datasets import load_dataset

raw_dataset = load_dataset("glue","mrpc")
# a dataset dict object with train validation test
raw_dataset.shape
# 4 ccolumns

{'train': (3668, 4), 'validation': (408, 4), 'test': (1725, 4)}

In [31]:
raw_train_dataset = raw_dataset['train']
raw_train_dataset[0]
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

In [32]:
# converting strings to numbers using tokenizers
# but working with paired data, need pass both pairs into tokenizer so it understands
# and puts it together 
# start with cls token and separate 2 pairs with sep tokens

from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)
tokenized_datasets
# use map to get back the dataset class structure

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [33]:
# converting strings to numbers using tokenizers
# but working with paired data, need pass both pairs into tokenizer so it understands
# and puts it together 
# start with cls token and separate 2 pairs with sep tokens

from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)
tokenized_datasets
# use map to get back the dataset class structure

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [34]:
# DYNAMIC PADDING using collator
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}

batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}
# padding to batch maximum reduces unnecesary cokmputation


{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}

In [35]:
# trainer class helps to fine tune on pretrained models
from transformers import TrainingArguments
# contains hypermeters the trainer use
training_args = TrainingArguments("test-trainer")

In [36]:
# trainer api -> helper that runs the rtraining loop 
# replaccee for epoch in epoch and train_loader
from transformers import Trainer
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In [37]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)
trainer.train()

  0%|          | 0/1377 [00:00<?, ?it/s]

{'loss': 0.5327, 'grad_norm': 2.921182155609131, 'learning_rate': 3.184458968772695e-05, 'epoch': 1.09}
{'loss': 0.3189, 'grad_norm': 143.69140625, 'learning_rate': 1.3689179375453886e-05, 'epoch': 2.18}
{'train_runtime': 81.236, 'train_samples_per_second': 135.457, 'train_steps_per_second': 16.951, 'train_loss': 0.3677013067551601, 'epoch': 3.0}


TrainOutput(global_step=1377, training_loss=0.3677013067551601, metrics={'train_runtime': 81.236, 'train_samples_per_second': 135.457, 'train_steps_per_second': 16.951, 'total_flos': 405114969714960.0, 'train_loss': 0.3677013067551601, 'epoch': 3.0})

In [38]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

  0%|          | 0/51 [00:00<?, ?it/s]

(408, 2) (408,)


In [39]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

In [40]:
import evaluate

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8455882352941176, 'f1': 0.8934010152284264}

In [41]:
# prepare for training by defining
# dataloaders, postprocessing to tokenized datasaets
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [43]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(tokenized_datasets["train"],shuffle=True,batch_size = 8,num_workers=2,collate_fn = data_collator)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

In [44]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}


{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 61]),
 'token_type_ids': torch.Size([8, 61]),
 'attention_mask': torch.Size([8, 61])}

In [48]:
device = "cuda"
batch = {k: v.to(device) for k, v in batch.items()}
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)


tensor(0.0056, device='cuda:0', grad_fn=<NllLossBackward0>) torch.Size([8, 2])


In [49]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(),lr=5e-5)

In [50]:
from transformers import get_scheduler
num_epochs = 3
num_trainig_steps = num_epochs*len(train_dataloader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_training_steps=1,
    num_warmup_steps=num_trainig_steps
)
print(num_trainig_steps)

1377


In [53]:
from tqdm.auto import tqdm


#tqdm used for progress bar
progress_bar = tqdm(range(num_trainig_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k:v.to(device) for k,v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)


  0%|          | 0/1377 [00:00<?, ?it/s]